# Menu ChatBot

In [1]:
from core.menu_parser import MenuParser
from exploratory_menu_analysis import *

# function to print section headers in the report
REPORT_WIDTH = 70
def print_section(title: str) -> None:
    print("\n" + "=" * REPORT_WIDTH)
    print(title)
    print("=" * REPORT_WIDTH)

This notebook walks through the implementation of the chatbot, and explores the data structure behind it, its limitations, and possible improvements.

## Menu Parsing and exploratory data analysis

The first step is to transform the raw menu JSON into a structured format that is easier to work with.

The `MenuParser` is designed as a consistent abstraction layer over the raw JSON data, organizing it into indexed structures that support efficient and reliable access.

A key design decision was to centralize normalization (for item names and sizes), ensuring that all lookups follow the same rules across the system. This avoids duplicating normalization logic in downstream components.

The parser builds multiple access patterns (by name, ID, and category), each serving a clear purpose. It also explicitly handles edge cases such as name collisions, making the system more robust.

In addition, the parser exposes a simple API (`get_item`, `get_price`) with a consistent response format, including structured error handling.

While the parser is intentionally simple, it already includes some query-related logic (e.g., price validation). In a larger system, this responsibility could be further separated, but for this scope it helps keep the overall design compact and practical.

In [2]:
parser = MenuParser('data/MenuDataTest.json')

# Lookup example
parser.get_item("nutty bowl")

# Price queries
parser.get_price("nutty bowl", "small")

{'success': True, 'data': {'price': 11.99, 'size': 'small'}, 'error': None}

## Exploratory Data Analysis


A quick exploratory analysis was performed to understand the structure and quality of the dataset.

In [3]:
run_exploratory_analysis()


1. DATA INVENTORY
Total raw item entries: 46
Total unique items by id: 46
Total indexed names: 46

Sample indexed item keys (first 10):
  1. dragon bowl
  2. superfood bowl
  3. warrior bowl
  4. green bowl
  5. nutty bowl
  6. tropical bowl
  7. kids bowl
  8. dessert bowl
  9. acai elixir
  10. go green

Name collisions: 0

2. PRICE EDGE CASES
Price range: $0.00 - $16.99
Zero/negative prices: 8
  They may need to be updated or removed from the menu.
  Items: ['temptation', 'power panini', 'superseed avocado toast', 'pb  chia jam toast', 'wholesome hummus toast', 'grilled cheese', 'kids pbj', 'kids sunsation smoothie']
Single-price items: 31
Multi-price items: 15
  Examples single: ['kids bowl', 'dessert bowl', 'acai elixir']
  Examples multi: ['dragon bowl', 'superfood bowl', 'warrior bowl']

3. NORMALIZATION AND LOOKUP CHECKS
Name normalization checks:
  OK: 'NUTTY  BOWL' -> NUTTY BOWL
  OK: 'GO GREEN!' -> GO GREEN
  OK: 'go green' -> GO GREEN
  OK: 'Go Green' -> GO GREEN
  OK: '  


The menu contains **46 unique items**, with consistent naming and no collisions after normalization, which enables reliable lookups.

Pricing is heterogeneous: most items have a single price, while others depend on size, requiring flexible handling of size-based queries. A few edge cases (e.g., zero-priced items) are present and, while acceptable for this exercise, would require careful validation in a production setting.

Nutrition data is sparse, with only a small subset of items providing this information, while discounts are more widely available and often linked to multiple items.

The menu is organized into a small number of categories, which supports grouping and filtering queries.

Overall, the dataset is well-structured, but its variability (in pricing, nutrition availability, and discount rules) directly influences how queries need to be handled.

## Chunking and RAG Pipeline



The system uses four specialized chunkers to transform structured data into retrieval-friendly text chunks:
- **ItemChunker**: Individual items with prices
- **CategoryChunker**: Category overviews
- **NutritionChunker**: Nutrition info (calories, dietary notes)
- **DiscountChunker**: Discount codes and eligibility

Each chunk is embedded and stored in ChromaDB for semantic retrieval.

In [3]:
from rag.chunkers.chunk_builder import ChunkBuilder

chunk_builder = ChunkBuilder(parser)
chunks, metadatas, ids = chunk_builder.build_all()

print(f"Total chunks created: {len(chunks)}")
print(f"\nChunk breakdown by type:")
from collections import Counter
types = Counter([m["type"] for m in metadatas])
for chunk_type, count in sorted(types.items()):
    print(f"  {chunk_type}: {count}")

print(f"\nExample item chunk:")
print(chunks[0])
print(f"\nExample discount chunk:")
discount_idx = next(i for i, m in enumerate(metadatas) if m["type"] == "discount")
print(chunks[discount_idx])

Total chunks created: 63

Chunk breakdown by type:
  category: 7
  discount: 7
  item: 46
  nutrition: 3

Example item chunk:
DRAGON BOWL is a menu item in the category: acai bowls.
            The item "DRAGON BOWL" has the following prices:
    - Medium: $14.49
- Large: $15.99

Example discount chunk:
Discount: 2 SM Bowls for $20

$20 off.


## Query Executor: Structured Lookups



For exact queries (prices, calories, categories), the system bypasses the LLM and uses deterministic lookups to ensure accuracy.


In [4]:
from core.query_executor import QueryExecutor

executor = QueryExecutor(parser)

# Price lookup
price_result = executor.execute({
    "intent": "item_price",
    "item_name": "nutty bowl",
    "size": "small",
    "category": None,
})
print(f"Price query result: {price_result}")

# Nutrition lookup
nutrition_result = executor.execute({
    "intent": "item_nutrition",
    "item_name": "go green",
    "size": None,
    "category": None,
})
print(f"Nutrition query result: {nutrition_result}")

# Category list
category_result = executor.execute({
    "intent": "category_list",
    "category": "salads",
    "item_name": None,
    "size": None,
})
print(f"Category query result:\n{category_result}")

Price query result: $11.99 (small)
Nutrition query result: GO GREEN is a smoothie with 240 calories.
Category query result:
Salads:
- supergreen goddess salad
- mighty med salad
- chimichurri steak  pot bowl
- green glow bowl
- power pesto chicken bowl


## Full Hybrid System



The RAGEngine orchestrates everything:
1. Intent detection (is this a price query? discount query? semantic query?)
2. Retrieval from vector store (using ChromaDB with optional type filtering)
3. QueryExecutor fallback (for structured intents)
4. LLM generation (for natural language synthesis)

Challenge questions demonstrate the full pipeline:

In [5]:
from rag.embeddings import EmbeddingModel
from rag.vector_store import VectorStore
from rag.llm_client import OllamaClient
from rag.rag_engine import RAGEngine

print("Building RAG system...")

# Embeddings and vector store
embedding_model = EmbeddingModel()
embeddings = embedding_model.embed_documents(chunks)

vector_store = VectorStore()
vector_store.add(chunks, embeddings, metadatas, ids)

# LLM and RAG engine
llm = OllamaClient()
rag = RAGEngine(embedding_model, vector_store, llm, executor)

print("RAG system ready.\n")

# Challenge questions
challenge_questions = [
    "What is the price of a small NUTTY BOWL?",
    "How many calories does the GO GREEN smoothie have?",
    "Which salads do you have?",
    "What discounts are available today?",
    "Which discounts include coupons?",
    "What items trigger a BOGO Any Smoothie discount?",
    "Is the price for Smoothie - ACAI ELIXIR the same in all channels?"
]

for i, question in enumerate(challenge_questions, 1):
    print(f"Q{i}: {question}")
    response = rag.run(question)
    print(f"A: {response}\n")

Building RAG system...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RAG system ready.

Q1: What is the price of a small NUTTY BOWL?
A: $11.99 (small)

Q2: How many calories does the GO GREEN smoothie have?
A: GO GREEN is a smoothie with 240 calories.

Q3: Which salads do you have?
A: We have the Supergreen Goddess Salad, MIGHTY MED SALAD, Green Glow Bowl, Power Pesto Chicken Bowl, and Chimichurri Steak & Pot Bowl. Enjoy!

Q4: What discounts are available today?
A: Active discounts today:
- $1 OFF any Bowl or Smoothie ($1 off)
- $3.00 Off LG Bowl ($3 off)
- 2 SM Bowls for $20 ($20 off)
- BOGO 50% off Wrap or Salad (50% off)
- BOGO Any Smoothie (100% off)
- Military/Police Discount (20% off)
- Veteran Day Smoothie (100% off)

Q5: Which discounts include coupons?
A: Available coupon codes:
- 1000: $3.00 Off LG Bowl
- Military: Military/Police Discount

Q6: What items trigger a BOGO Any Smoothie discount?
A: Any Smoothie!

Q7: Is the price for Smoothie - ACAI ELIXIR the same in all channels?
A: I only have one menu snapshot and cannot compare prices across

## More Query Examples

Try different kinds of queries to see how the system handles them:

In [ ]:
# Additional query examples
more_queries = [
    "What's the price of a large smoothie?",
    "Tell me about the acai elixir",
    "What items have protein in them?",
    "What bowls do you have?",
    "Are there any vegetarian options?",
    "What's the cheapest item on the menu?",
    "Do you have any cold beverages?",
    "What's in the power pesto bowl?",
]

print("Additional query examples:\n")
for question in more_queries:
    print(f"Q: {question}")
    response = rag.run(question)
    print(f"A: {response}\n")

Additional query examples:

Q: What's the price of a large smoothie?
A: Please specify the smoothie name with size 'large'. Examples: ACAI ELIXIR, GO GREEN, MATCHA MADNESS, TROPICAL PARADISE, DRAGON SMOOTHIE.

Q: Tell me about the acai elixir
A: The ACai Elixir is a smoothie that costs $8.49 by default!

Q: What items have protein in them?
A: Based on the menu information, the items with protein in them are:
* CHIMICHURRI STEAK WRAP
* MORNING GLORY WRAP
* POWER PANINI (since it's a panini, likely containing meat or eggs)
* WHEY GREEN (a smoothie with an unspecified amount of whey protein)
Please note that the menu doesn't provide specific information on the protein content of some items. These are the items where protein is explicitly mentioned or can be inferred from the name.

Q: What bowls do you have?
A: We have WARRIOR BOWL, GREEN BOWL, DRAGON BOWL, NUTTY BOWL, TROPICAL BOWL, KIDS BOWL, and DESSERT BOWL!

Q: Are there any vegetarian options?
A: Yes, there are several vegetarian op